In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
model_id = "Qwen/Qwen3-0.6B"

# Auto-detect device: CUDA > MPS > CPU
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16  # MPS doesn't fully support bfloat16
else:
    device = "cpu"
    dtype = torch.float32

print(f"Device: {device} | dtype: {dtype}")

Device: cuda | dtype: torch.bfloat16


In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=dtype,
    device_map=device
)
print(model.config)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 40960,
  "max_w

In [4]:
print(f"Model: {model.config.model_type}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Dtype: {next(model.parameters()).dtype}")

Model: qwen3
Parameters: 596,049,920
Dtype: torch.bfloat16


In [5]:
messages = [
    {
        "role": "user",
        "content": "Explain transformers in one sentence.",
    }
]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

print(f"\nPrompt tokens: {inputs['input_ids'].shape[1]}")

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
    )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :],
        skip_special_tokens=True,
    )

    print(response)


Prompt tokens: 15
<think>
Okay, the user wants me to explain Transformers in one sentence. Let me start by recalling what I know about Transformers.

Transformers are a type of neural network that uses self-attention mechanisms to process information more efficiently. They're used in tasks like language modeling and text generation. So combining these elements into one sentence would be important.

I need to make sure it's concise but covers their key components: self-attention, efficient processing, applications in AI. Let me check if there's any other aspect I should mention. Maybe mention how they handle long sequences? But since it's one sentence, maybe stick to the main points. Also, confirm that the sentence flows well without redundancy.
</think>

Transformers use self-attention mechanisms to process sequential data efficiently, enabling them to model complex dependencies in natural language tasks.


In [6]:
# --- Inspect the model structure ---
print(f"\nModel layers (top-level):")
for name, module in model.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f"  {name}: {type(module).__name__} ({n_params:,} params)")


Model layers (top-level):
  model: Qwen3Model (596,049,920 params)
  lm_head: Linear (155,582,464 params)


In [7]:
for name, module in model.model.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f"  {name}: {type(module).__name__} ({n_params:,} params)")

  embed_tokens: Embedding (155,582,464 params)
  layers: ModuleList (440,466,432 params)
  norm: Qwen3RMSNorm (1,024 params)
  rotary_emb: Qwen3RotaryEmbedding (0 params)
